# 06 — Deployment

Notebook ini menjalankan tahap **Deployment** CRISP-DM untuk klasifikasi biner audio:

- `1 = berisik`
- `0 = non-berisik`

Tujuan tahap ini adalah membangun **`predict_audio(path)`** yang melakukan prapemrosesan identik dengan Tahap 03, memakai model tersimpan dari Tahap 04 **tanpa training ulang**, mendemokan fungsi pada file asli dataset, dan menulis laporan akhir `artifacts/LAPORAN.md` berisi data aktual, fitur, model, metrik nyata, lokasi artefak, dan limitasi.

## Masukan

- `artifacts/best_model.joblib` dan `artifacts/04_modeling/model_config.json`.
- `artifacts/03_data_preparation/preprocessing_contract.json`: kontrak prapemrosesan yang wajib diikuti.
- `artifacts/features.csv` dan `artifacts/03_data_preparation/transformation_manifest.csv`.
- `artifacts/04_modeling/holdout_file_split.csv` dan `artifacts/test_results/` dari Tahap 05.
- `data_mp3/` dan `data_final/` untuk demo.

## Keluaran utama

- Fungsi `predict_audio(path)` di notebook ini (cell `08_predict_audio.py`).
- `artifacts/06_deployment/feature_identity_check.csv` dan `identity_summary.json`: bukti prapemrosesan identik untuk 176 file.
- `artifacts/06_deployment/demo_results.csv`: hasil demo nyata (dua kelas) plus uji cabang crop/pad.
- `artifacts/LAPORAN.md`: laporan akhir proyek.
- `artifacts/06_deployment/`: laporan dan validasi tahap.

## Batasan yang dipegang

1. Model **tidak** di-fit ulang, tidak ada penalaan threshold, tidak ada pemilihan fitur baru.
2. Probabilitas keluaran adalah skor model, bukan keyakinan terkalibrasi.
3. Syarat kelayakan awal ROC-AUC ≥ 0,85 dilaporkan **apa adanya** dari Tahap 05.
4. Demo memakai file asli dataset; seluruh angka berasal dari eksekusi nyata; `raw_audio/` tidak diubah.


In [1]:
import hashlib
import json
import pickle
import platform
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found.*")

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 120)

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42

library_versions = pd.DataFrame(
    {
        "komponen": [
            "Python",
            "numpy",
            "pandas",
            "scikit-learn",
            "joblib",
            "librosa",
            "soundfile",
            "matplotlib",
        ],
        "versi": [
            platform.python_version(),
            np.__version__,
            pd.__version__,
            __import__("sklearn").__version__,
            joblib.__version__,
            librosa.__version__,
            sf.__version__,
            plt.matplotlib.__version__,
        ],
    }
)

display(library_versions)


,komponen,versi
0,Python,3.11.9
1,numpy,2.4.6
2,pandas,2.3.3
3,scikit-learn,1.8.0
4,joblib,1.5.3
5,librosa,0.11.0
6,soundfile,0.14.0
7,matplotlib,3.11.0


In [2]:
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "raw_audio").is_dir() and (candidate / "artifacts").is_dir():
        PROJECT_ROOT = candidate.resolve()
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Folder proyek dengan raw_audio/ dan artifacts/ tidak ditemukan.")

RAW_DIR = PROJECT_ROOT / "raw_audio"
DATA_MP3_DIR = PROJECT_ROOT / "data_mp3"
DATA_FINAL_DIR = PROJECT_ROOT / "data_final"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR = ARTIFACT_ROOT / "06_deployment"
DEMO_AUDIO_DIR = ARTIFACT_DIR / "demo_audio"
LAPORAN_PATH = ARTIFACT_ROOT / "LAPORAN.md"

FEATURES_PATH = ARTIFACT_ROOT / "features.csv"
MODEL_PATH = ARTIFACT_ROOT / "best_model.joblib"
MODEL_CONFIG_PATH = ARTIFACT_ROOT / "04_modeling" / "model_config.json"
HOLDOUT_FILE_PATH = ARTIFACT_ROOT / "04_modeling" / "holdout_file_split.csv"
TRANSFORMATION_MANIFEST_PATH = (
    ARTIFACT_ROOT / "03_data_preparation" / "transformation_manifest.csv"
)
PREPROCESSING_CONTRACT_PATH = (
    ARTIFACT_ROOT / "03_data_preparation" / "preprocessing_contract.json"
)
STAGE3_VALIDATION_PATH = ARTIFACT_ROOT / "03_data_preparation" / "validation_summary.json"
STAGE4_VALIDATION_PATH = ARTIFACT_ROOT / "04_modeling" / "validation_summary.json"
STAGE5_VALIDATION_PATH = ARTIFACT_ROOT / "05_evaluation" / "validation_summary.json"
STAGE3_SUMMARY_PATH = ARTIFACT_ROOT / "03_data_preparation" / "data_preparation_summary.json"

PREDICTIONS_PATH = ARTIFACT_ROOT / "test_results" / "test_predictions.csv"
METRICS_PATH = ARTIFACT_ROOT / "test_results" / "metrics.json"
EVALUATION_SUMMARY_PATH = ARTIFACT_ROOT / "05_evaluation" / "evaluation_summary.json"

IDENTITY_CHECK_PATH = ARTIFACT_DIR / "feature_identity_check.csv"
IDENTITY_SUMMARY_PATH = ARTIFACT_DIR / "identity_summary.json"
DEMO_RESULTS_PATH = ARTIFACT_DIR / "demo_results.csv"

for directory in [ARTIFACT_DIR, DEMO_AUDIO_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SR = 16000
TARGET_DURATION_S = 30.0
TARGET_SAMPLES = int(SR * TARGET_DURATION_S)
FRAME_LENGTH = 400
HOP_LENGTH = 160
N_FFT = 512
ROLLOFF_PERCENT = 0.85
TARGET_PEAK = 1.0
EPS = np.finfo(np.float64).eps

IDENTITY_COLUMNS = ["file", "nim", "take", "label"]

FEATURE_COLUMNS = [
    "mse_energy",
    "rms_mean",
    "rms_std",
    "rms_max",
    "rms_min",
    "zcr_mean",
    "zcr_std",
    "spectral_centroid_mean",
    "spectral_centroid_std",
    "spectral_flatness_mean",
    "spectral_rolloff_mean",
    "spectral_bandwidth_mean",
]

LABEL_NAME_MAP = {0: "non-berisik", 1: "berisik"}
LABEL_ORDER = ["non-berisik", "berisik"]

TEST_THRESHOLD = 0.5
FEATURE_REL_TOL = 1e-3
PROBA_MATCH_TOL = 5e-3
MODEL_IDENTITY_TOL = 1e-12

CONFIG = pd.DataFrame(
    {
        "parameter": [
            "project_root",
            "model",
            "threshold keputusan",
            "toleransi fitur relatif",
            "toleransi probabilitas",
            "sample rate",
            "target sampel",
            "frame / hop / n_fft",
            "laporan akhir",
        ],
        "nilai": [
            str(PROJECT_ROOT),
            str(MODEL_PATH),
            TEST_THRESHOLD,
            FEATURE_REL_TOL,
            PROBA_MATCH_TOL,
            SR,
            TARGET_SAMPLES,
            f"{FRAME_LENGTH} / {HOP_LENGTH} / {N_FFT}",
            str(LAPORAN_PATH),
        ],
    }
)

display(CONFIG)


,parameter,nilai
0,project_root,C:\Users\ASUS\Downloads\dataset_30_detik
1,model,C:\Users\ASUS\Downloads\dataset_30_detik\artif...
2,threshold keputusan,0.5
3,toleransi fitur relatif,0.001
4,toleransi probabilitas,0.005
5,sample rate,16000
6,target sampel,480000
7,frame / hop / n_fft,400 / 160 / 512
8,laporan akhir,C:\Users\ASUS\Downloads\dataset_30_detik\artif...


## 1. Pintu masuk: artefak tanpa training ulang

Section ini memuat seluruh artefak yang dibutuhkan deployment dan membuktikan bahwa model yang dipakai adalah model yang sama persis dengan yang diuji pada Tahap 05.

Syarat yang diverifikasi:

1. `validation_summary.json` Tahap 03, 04, dan 05 semuanya berstatus **PASS** dengan seluruh pemeriksaan lulus.
2. `best_model.joblib` ada, pipeline `scaler -> select -> clf`, seluruh parameter cocok dengan `model_config.json`, dan 12 kolom fitur tidak berubah.
3. Kontrak prapemrosesan `preprocessing_contract.json` (16 kHz, 480.000 sampel, frame 400, hop 160, n_fft 512, rolloff 0,85) cocok dengan konstanta notebook ini.
4. **Identitas model**: `predict_proba` langsung pada baris `features.csv` untuk 40 file uji menghasilkan probabilitas identik (selisih < 1e-12) dengan `test_predictions.csv` Tahap 05 — bukti model tidak diganti maupun di-fit ulang.

Sidik jari SHA-256 file model dan sidik jari pickle pipeline dicatat saat dimuat untuk laporan dan validasi akhir.


In [3]:
stage3_validation = json.loads(STAGE3_VALIDATION_PATH.read_text(encoding="utf-8"))
stage4_validation = json.loads(STAGE4_VALIDATION_PATH.read_text(encoding="utf-8"))
stage5_validation = json.loads(STAGE5_VALIDATION_PATH.read_text(encoding="utf-8"))

for stage_name, summary in {
    "03": stage3_validation,
    "04": stage4_validation,
    "05": stage5_validation,
}.items():
    assert summary["status"] == "PASS", f"Validasi Tahap {stage_name} bukan PASS."
    assert summary["checks_passed"] == summary["checks_total"], (
        f"Ada pemeriksaan gagal pada Tahap {stage_name}."
    )
assert stage4_validation["holdout_tested_in_stage_04"] is False, (
    "Hold-out sudah pernah diuji di Tahap 04."
)

assert MODEL_PATH.is_file(), "best_model.joblib tidak ditemukan."
model_config = json.loads(MODEL_CONFIG_PATH.read_text(encoding="utf-8"))
saved_model = joblib.load(MODEL_PATH)

assert list(saved_model.named_steps) == ["scaler", "select", "clf"], (
    f"Struktur pipeline berubah: {list(saved_model.named_steps)}"
)
assert model_config["model_name"] == "Logistic Regression"
assert model_config["feature_columns"] == FEATURE_COLUMNS, "Daftar fitur config tidak cocok."
assert int(saved_model.n_features_in_) == len(FEATURE_COLUMNS)

for param_key, param_value in model_config["best_params"].items():
    assert saved_model.get_params()[param_key] == param_value, f"Parameter berubah: {param_key}"

selected_from_model = list(
    saved_model.named_steps["select"].get_feature_names_out(FEATURE_COLUMNS)
)
assert selected_from_model == model_config["selected_features_final"], (
    "Fitur terpilih pada pipeline berubah."
)
assert int(saved_model.named_steps["select"].k) == int(model_config["best_params"]["select__k"])
assert list(saved_model.classes_) == [0, 1], "Urutan kelas model berubah."

contract = json.loads(PREPROCESSING_CONTRACT_PATH.read_text(encoding="utf-8"))
assert contract["sample_rate"] == SR, "Sample rate kontrak berubah."
assert contract["target_samples"] == TARGET_SAMPLES, "Jumlah sampel kontrak berubah."
assert contract["frame_length"] == FRAME_LENGTH, "Frame length kontrak berubah."
assert contract["hop_length"] == HOP_LENGTH, "Hop length kontrak berubah."
assert contract["n_fft"] == N_FFT, "n_fft kontrak berubah."
assert contract["rolloff_percent"] == ROLLOFF_PERCENT, "Rolloff kontrak berubah."
assert float(contract["flatness_epsilon"]) == float(EPS), "Epsilon flatness kontrak berubah."

model_file_sha256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
model_fingerprint = hashlib.sha256(pickle.dumps(saved_model)).hexdigest()
assert len(model_file_sha256) == 64 and len(model_fingerprint) == 64

features = pd.read_csv(FEATURES_PATH)
features["label"] = features["label"].astype(int)
features["take"] = features["take"].astype(int)
assert len(features) == 176, f"features.csv berisi {len(features)} baris."
assert np.isfinite(features[FEATURE_COLUMNS].to_numpy(dtype=float)).all(), "Fitur mengandung NaN/Inf."

holdout_file_split = pd.read_csv(HOLDOUT_FILE_PATH)
assert len(holdout_file_split) == 176 and set(holdout_file_split["split"]) == {"train", "test"}

transformation_manifest = pd.read_csv(TRANSFORMATION_MANIFEST_PATH)
assert len(transformation_manifest) == 176
assert transformation_manifest["file"].is_unique, "Manifest transformasi memiliki file ganda."
assert set(transformation_manifest["file"]) == set(features["file"]), (
    "Manifest transformasi dan features.csv tidak seisi."
)

test_predictions = pd.read_csv(PREDICTIONS_PATH)
assert len(test_predictions) == 40, "test_predictions.csv tidak berisi 40 baris."
assert set(test_predictions["split"]) == {"test"}

metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
evaluation_summary = json.loads(EVALUATION_SUMMARY_PATH.read_text(encoding="utf-8"))
stage3_summary = json.loads(STAGE3_SUMMARY_PATH.read_text(encoding="utf-8"))
assert int(metrics["n_files_test"]) == 40

nims_test = sorted(model_config["holdout"]["nims_test"])
nims_train = sorted(model_config["holdout"]["nims_train"])
assert len(nims_test) == 7 and len(nims_train) == 27

X_test_direct = features.set_index("file").loc[test_predictions["file"], FEATURE_COLUMNS]
proba_direct = saved_model.predict_proba(X_test_direct.to_numpy(dtype=float))[:, 1]
model_identity_max_diff = float(np.max(np.abs(proba_direct - test_predictions["proba_berisik"])))
assert model_identity_max_diff < MODEL_IDENTITY_TOL, (
    f"Probabilitas model berbeda dari Tahap 05: {model_identity_max_diff}"
)
pred_direct = (proba_direct >= TEST_THRESHOLD).astype(int)
assert int((pred_direct != test_predictions["pred_label"].to_numpy()).sum()) == 0

gate_summary = pd.DataFrame(
    {
        "pemeriksaan": [
            "validasi Tahap 03 / 04 / 05",
            "model",
            "pipeline",
            "parameter terbaik",
            "fitur input",
            "fitur terpilih",
            "kontrak prapemrosesan",
            "selisih probabilitas vs Tahap 05",
            "sidik jari file model (sha256)",
            "NIM test",
            "file uji",
        ],
        "nilai": [
            "PASS / PASS / PASS",
            model_config["model_name"],
            " -> ".join(list(saved_model.named_steps)),
            str(model_config["best_params"]),
            len(FEATURE_COLUMNS),
            len(selected_from_model),
            f"{SR} Hz / {TARGET_SAMPLES} sampel / frame {FRAME_LENGTH} / hop {HOP_LENGTH}",
            f"{model_identity_max_diff:.3e}",
            model_file_sha256[:16],
            ", ".join(nims_test),
            len(test_predictions),
        ],
    }
)
display(gate_summary)
print("Pintu masuk Tahap 06: LULUS — model identik dengan yang dievaluasi Tahap 05.")


,pemeriksaan,nilai
0,validasi Tahap 03 / 04 / 05,PASS / PASS / PASS
1,model,Logistic Regression
2,pipeline,scaler -> select -> clf
3,parameter terbaik,"{'clf__C': 10.0, 'select__k': 8}"
4,fitur input,12
5,fitur terpilih,8
6,kontrak prapemrosesan,16000 Hz / 480000 sampel / frame 400 / hop 160
7,selisih probabilitas vs Tahap 05,1.110e-16
8,sidik jari file model (sha256),964a1550c9c06123
9,NIM test,"24-068, 24-100, 24-119, 24-131, 24-145, 24-168..."


Pintu masuk Tahap 06: LULUS — model identik dengan yang dievaluasi Tahap 05.


## 2. Prapemrosesan dan ekstraksi fitur identik Tahap 03

Deployment hanya benar bila file masuk diproses persis seperti data latih. Bagian ini mendefinisikan fungsi-fungsi berikut dengan rumus dan parameter yang sama persis dengan Tahap 03:

1. **Dekode** — `librosa.load(path, sr=16000, mono=True)`, sama seperti kontrak `preprocessing_contract.json`.
2. **Penyesuaian durasi** — crop tengah bila lebih panjang, zero-pad tengah bila lebih pendek, target tepat 480.000 sampel.
3. **Peak normalization** — `y / max(|y|) * 1.0` setelah crop/pad, hasil di-cast `float32`; sinyal dengan peak nol ditolak, bukan dinormalisasi.
4. **Ekstraksi 12 fitur** — framing `frame_length=400`, `hop=160` tanpa `center`, window Hann simetris, rFFT 512 titik, daya `P = |S|^2`, epsilon `np.finfo(np.float64).eps`, rolloff 0,85; seluruh formula disalin apa adanya dari sel Tahap 03.

Setiap fitur dihitung dari sinyal yang sama dengan yang dipakai `features.csv`, termasuk perlakuan frame daya nol (centroid, flatness, rolloff, dan bandwidth frame tersebut ditetapkan 0). Kelas `predict_audio` berikutnya membungkus fungsi-fungsi ini.


In [4]:
FREQS = np.fft.rfftfreq(N_FFT, d=1 / SR)
HANN_WINDOW = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(FRAME_LENGTH) / (FRAME_LENGTH - 1))
if not np.allclose(HANN_WINDOW, np.hanning(FRAME_LENGTH), atol=1e-15):
    raise ValueError("Window Hann tidak sesuai definisi simetris.")


def fit_duration_to_target(y, target_samples=TARGET_SAMPLES):
    y = np.asarray(y)
    original_samples = int(len(y))
    crop_start = 0
    pad_left = 0
    pad_right = 0

    if original_samples > target_samples:
        action = "crop"
        crop_start = (original_samples - target_samples) // 2
        y_out = y[crop_start : crop_start + target_samples].copy()
    elif original_samples < target_samples:
        action = "pad"
        total_pad = target_samples - original_samples
        pad_left = total_pad // 2
        pad_right = total_pad - pad_left
        y_out = np.pad(y, (pad_left, pad_right), mode="constant", constant_values=0.0)
    else:
        action = "none"
        y_out = y.copy()

    if len(y_out) != target_samples:
        raise ValueError("Panjang sinyal tidak sama dengan target sampel.")

    return y_out, {
        "original_samples_16k": original_samples,
        "original_duration_16k_s": original_samples / SR,
        "action": action,
        "crop_start": int(crop_start),
        "pad_left": int(pad_left),
        "pad_right": int(pad_right),
        "final_samples": int(len(y_out)),
    }


def peak_normalize(y, target_peak=TARGET_PEAK):
    y = np.asarray(y)
    if y.size == 0:
        raise ValueError("Sinyal kosong tidak dapat dinormalisasi.")
    if not np.isfinite(y).all():
        raise ValueError("Sinyal mengandung NaN atau Inf.")
    peak_before = float(np.max(np.abs(y)))
    if peak_before <= 0:
        raise ValueError("Peak sinyal nol; file harus ditolak, bukan dinormalisasi.")
    y_norm = (y / peak_before * target_peak).astype(np.float32)
    peak_after = float(np.max(np.abs(y_norm)))
    return y_norm, {
        "peak_before": peak_before,
        "peak_after": peak_after,
        "target_peak": float(target_peak),
    }


def frame_signal(y, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH):
    y = np.asarray(y)
    if len(y) < frame_length:
        raise ValueError("Sinyal lebih pendek dari satu frame.")
    frames = np.lib.stride_tricks.sliding_window_view(y, frame_length)[::hop_length]
    return frames


def extract_features(y):
    y = np.asarray(y)
    if len(y) != TARGET_SAMPLES:
        raise ValueError(f"Fitur hanya boleh dihitung dari {TARGET_SAMPLES} sampel.")
    if not np.isfinite(y).all():
        raise ValueError("Sinyal mengandung NaN atau Inf.")

    features = {"mse_energy": float(np.mean(y.astype(np.float64) ** 2))}

    frames = frame_signal(y).astype(np.float64)
    rms_frames = np.sqrt(np.mean(frames**2, axis=1))
    zcr_frames = np.mean((frames[:, :-1] * frames[:, 1:]) < 0, axis=1)

    P = np.abs(np.fft.rfft(frames * HANN_WINDOW[None, :], n=N_FFT, axis=1)) ** 2
    power_sum = P.sum(axis=1)
    centroids = np.divide(
        P @ FREQS,
        power_sum,
        out=np.zeros_like(power_sum),
        where=power_sum > 0,
    )

    P_eps = P + EPS
    flatness_frames = np.exp(np.mean(np.log(P_eps), axis=1)) / np.mean(P_eps, axis=1)
    flatness_frames[P.sum(axis=1) == 0] = 0

    cumulative = np.cumsum(P, axis=1)
    total_power = cumulative[:, -1]
    thresholds = ROLLOFF_PERCENT * total_power
    rolloff_indices = np.argmax(cumulative >= thresholds[:, None], axis=1)
    rolloff_frames = FREQS[rolloff_indices].copy()
    rolloff_frames[total_power == 0] = 0

    variance = np.divide(
        (P * (FREQS[None, :] - centroids[:, None]) ** 2).sum(axis=1),
        power_sum,
        out=np.zeros_like(power_sum),
        where=power_sum > 0,
    )
    bandwidth_frames = np.sqrt(np.maximum(variance, 0))

    features.update(
        {
            "rms_mean": float(np.mean(rms_frames)),
            "rms_std": float(np.std(rms_frames, ddof=0)),
            "rms_max": float(np.max(rms_frames)),
            "rms_min": float(np.min(rms_frames)),
            "zcr_mean": float(np.mean(zcr_frames)),
            "zcr_std": float(np.std(zcr_frames, ddof=0)),
            "spectral_centroid_mean": float(np.mean(centroids)),
            "spectral_centroid_std": float(np.std(centroids, ddof=0)),
            "spectral_flatness_mean": float(np.mean(flatness_frames)),
            "spectral_rolloff_mean": float(np.mean(rolloff_frames)),
            "spectral_bandwidth_mean": float(np.mean(bandwidth_frames)),
        }
    )

    if list(features) != FEATURE_COLUMNS:
        raise ValueError(f"Urutan fitur berubah: {list(features)}")
    if not all(np.isfinite(value) for value in features.values()):
        raise ValueError("Fitur mengandung NaN atau Inf.")
    return features


definition_table = pd.DataFrame(
    {
        "langkah": [
            "dekode",
            "durasi",
            "normalisasi",
            "frame",
            "window",
            "fft",
            "jumlah fitur",
        ],
        "definisi": [
            contract["decode"],
            f"crop/pad tengah ke {TARGET_SAMPLES} sampel",
            contract["peak_normalization"],
            f"frame_length={FRAME_LENGTH}, hop_length={HOP_LENGTH}, tanpa center",
            "Hann simetris L=400",
            f"rfft n={N_FFT} ({FREQS.size} bin, resolusi {SR / N_FFT:.2f} Hz)",
            len(FEATURE_COLUMNS),
        ],
    }
)
display(definition_table)
print("Fungsi prapemrosesan dan ekstraksi fitur siap.")


,langkah,definisi
0,dekode,"librosa.load(path, sr=16000, mono=True)"
1,durasi,crop/pad tengah ke 480000 sampel
2,normalisasi,after crop/pad; y/max(abs(y))*1.0; error if ma...
3,frame,"frame_length=400, hop_length=160, tanpa center"
4,window,Hann simetris L=400
5,fft,"rfft n=512 (257 bin, resolusi 31.25 Hz)"
6,jumlah fitur,12


Fungsi prapemrosesan dan ekstraksi fitur siap.


## 3. Fungsi `predict_audio(path)`

Fungsi ini adalah keluaran utama tahap deployment. Alurnya, sesuai rincian Tahap 06 di `01_business_understanding.md`:

1. memuat file audio apa pun (MP3 maupun WAV) lewat `librosa.load(..., sr=16000, mono=True)`;
2. menyetel durasi ke tepat 30 detik (crop/pad tengah);
3. melakukan peak normalization dengan aturan yang sama seperti training;
4. mengekstraksi 12 fitur dengan definisi dan parameter yang sama;
5. memakai `best_model.joblib` yang sudah dimuat (pipeline menangani scaling dan seleksi fitur di dalamnya);
6. mengembalikan **label prediksi** dan **probabilitas kelas yang diprediksi** beserta probabilitas kedua kelas, catatan durasi, dan nilai fiturnya.

Catatan yang dipegang:

- Threshold keputusan tetap **0,50** — konsisten dengan `predict` bawaan pipeline dan dengan evaluasi Tahap 05; tidak ada penalaan.
- Probabilitas keluaran adalah **skor model**, bukan keyakinan terkalibrasi.
- Fungsi menolak file yang tidak ada, sinyal kosong, atau peak nol dengan pesan galat yang jelas, bukan menghasilkan angka semu.


In [5]:
def predict_audio(path, model=None, threshold=TEST_THRESHOLD):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"File audio tidak ditemukan: {path}")

    active_model = saved_model if model is None else model

    y, loaded_sr = librosa.load(path, sr=SR, mono=True)
    if loaded_sr != SR:
        raise ValueError(f"Sample rate hasil dekode {loaded_sr} bukan {SR}.")

    y_fit, fit_meta = fit_duration_to_target(y)
    y_norm, peak_meta = peak_normalize(y_fit)
    feature_values = extract_features(y_norm)

    frame = pd.DataFrame([[feature_values[column] for column in FEATURE_COLUMNS]], columns=FEATURE_COLUMNS)
    proba = active_model.predict_proba(frame.to_numpy(dtype=float))[0]
    proba_map = {int(cls): float(value) for cls, value in zip(active_model.classes_, proba)}
    proba_berisik = proba_map[1]
    pred_label = int(proba_berisik >= threshold)

    return {
        "path": str(path),
        "file": path.name,
        "label": pred_label,
        "label_name": LABEL_NAME_MAP[pred_label],
        "proba_non_berisik": proba_map[0],
        "proba_berisik": proba_berisik,
        "proba_diprediksi": max(proba_map.values()),
        "threshold": float(threshold),
        "duration_action": fit_meta["action"],
        "original_samples_16k": fit_meta["original_samples_16k"],
        "original_duration_16k_s": fit_meta["original_duration_16k_s"],
        "peak_before_norm": peak_meta["peak_before"],
        "features": feature_values,
    }


smoke_path = DATA_FINAL_DIR / transformation_manifest.sort_values("file").iloc[0]["final_relative_path"]
smoke_result = predict_audio(smoke_path)
smoke_view = pd.DataFrame(
    {
        "kunci": [key for key in smoke_result if key != "features"],
        "nilai": [value for key, value in smoke_result.items() if key != "features"],
    }
)
display(smoke_view)
print(f"predict_audio siap: {smoke_path.name} -> {smoke_result['label_name']} (p={smoke_result['proba_berisik']:.4f})")


,kunci,nilai
0,path,C:\Users\ASUS\Downloads\dataset_30_detik\data_...
1,file,24-003_berisik_1.wav
2,label,1
3,label_name,berisik
4,proba_non_berisik,0.290089
5,proba_berisik,0.709911
6,proba_diprediksi,0.709911
7,threshold,0.5
8,duration_action,none
9,original_samples_16k,480000


predict_audio siap: 24-003_berisik_1.wav -> berisik (p=0.7099)


## 4. Bukti identitas: 176 file, fitur dan prediksi

Klaim "prosedur training dan prediksi identik" harus dibuktikan, bukan ditegaskan. Section ini menjalankan `predict_audio` pada **seluruh 176 file `data_final/`** dan membandingkannya dengan `features.csv` dari Tahap 03:

1. **Fitur** — selisih relatif per fitur per file dibandingkan `features.csv`; seluruhnya wajib di bawah toleransi. Sumber perbedaan yang diperbolehkan hanya kuantisasi ulang PCM16 ketika file final dinormalisasi kembali (faktor ~3e-5).
2. **Probabilitas** — probabilitas dari `predict_audio` dibandingkan `predict_proba` langsung pada baris `features.csv` yang sama.
3. **Label** — label `predict_audio` harus sama dengan label pipeline langsung untuk semua file.
4. **Reproduksi Tahap 05** — untuk 40 file uji, label wajib identik dengan `test_predictions.csv` dan probabilitasnya berbeda paling banyak `PROBA_MATCH_TOL`.

Hasil per file disimpan di `feature_identity_check.csv` dan ringkasannya di `identity_summary.json`.


In [6]:
path_by_file = transformation_manifest.set_index("file")
split_by_file = holdout_file_split.set_index("file")["split"]
recorded = test_predictions.set_index("file")

identity_rows = []
for row in features.sort_values("file").itertuples():
    final_path = DATA_FINAL_DIR / path_by_file.at[row.file, "final_relative_path"]
    result = predict_audio(final_path)

    ref_features = {column: float(getattr(row, column)) for column in FEATURE_COLUMNS}
    rel_diffs = {}
    abs_diffs = {}
    for column in FEATURE_COLUMNS:
        pipeline_value = float(result["features"][column])
        reference_value = ref_features[column]
        abs_diffs[column] = abs(pipeline_value - reference_value)
        denom = max(abs(pipeline_value), abs(reference_value), 1e-12)
        rel_diffs[column] = abs_diffs[column] / denom

    frame_direct = pd.DataFrame([[ref_features[c] for c in FEATURE_COLUMNS]], columns=FEATURE_COLUMNS)
    proba_direct = float(saved_model.predict_proba(frame_direct.to_numpy(dtype=float))[0][1])
    pred_direct = int(proba_direct >= TEST_THRESHOLD)
    split_value = split_by_file.at[row.file]

    proba_recorded = float(recorded.at[row.file, "proba_berisik"]) if split_value == "test" else np.nan
    pred_recorded = int(recorded.at[row.file, "pred_label"]) if split_value == "test" else -1

    identity_rows.append(
        {
            "file": row.file,
            "nim": row.nim,
            "label": int(row.label),
            "split": split_value,
            "duration_action": result["duration_action"],
            "proba_direct": proba_direct,
            "proba_pipeline": result["proba_berisik"],
            "proba_recorded": proba_recorded,
            "pred_direct": pred_direct,
            "pred_pipeline": result["label"],
            "pred_recorded": pred_recorded,
            "proba_pipeline_vs_direct": abs(result["proba_berisik"] - proba_direct),
            "proba_pipeline_vs_recorded": (
                abs(result["proba_berisik"] - proba_recorded)
                if split_value == "test"
                else np.nan
            ),
            "feature_max_rel_diff": max(rel_diffs.values()),
            "feature_max_abs_diff": max(abs_diffs.values()),
            "feature_worst": max(rel_diffs, key=rel_diffs.get),
        }
    )

identity_check = pd.DataFrame(identity_rows)
assert len(identity_check) == 176
assert np.isfinite(
    identity_check[["proba_direct", "proba_pipeline", "feature_max_rel_diff"]].to_numpy()
).all()

max_feature_rel_diff = float(identity_check["feature_max_rel_diff"].max())
max_feature_abs_diff = float(identity_check["feature_max_abs_diff"].max())
max_proba_diff_direct = float(identity_check["proba_pipeline_vs_direct"].max())
n_label_match_176 = int((identity_check["pred_pipeline"] == identity_check["pred_direct"]).sum())

test_identity = identity_check.loc[identity_check["split"] == "test"]
assert len(test_identity) == 40
n_test_label_match = int((test_identity["pred_pipeline"] == test_identity["pred_recorded"]).sum())
max_test_proba_diff = float(test_identity["proba_pipeline_vs_recorded"].max())
test_accuracy = float(
    (identity_check["pred_pipeline"] == identity_check["label"]).loc[
        identity_check["split"] == "test"
    ].mean()
)

identity_summary = {
    "n_files_checked": int(len(identity_check)),
    "max_feature_rel_diff": max_feature_rel_diff,
    "max_feature_abs_diff": max_feature_abs_diff,
    "max_proba_diff_pipeline_vs_direct": max_proba_diff_direct,
    "label_match_vs_direct_176": n_label_match_176,
    "test_label_match_vs_tahap05": n_test_label_match,
    "test_max_proba_diff_vs_tahap05": max_test_proba_diff,
    "test_accuracy_pipeline": test_accuracy,
    "test_accuracy_tahap05": float(metrics["accuracy"]),
    "feature_rel_tolerance": FEATURE_REL_TOL,
    "proba_match_tolerance": PROBA_MATCH_TOL,
}
IDENTITY_SUMMARY_PATH.write_text(
    json.dumps(identity_summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
identity_check.to_csv(IDENTITY_CHECK_PATH, index=False)

identity_view = pd.DataFrame(
    {
        "pemeriksaan": [
            "file diperiksa",
            "selisih fitur relatif maksimum",
            "selisih fitur absolut maksimum",
            "selisih probabilitas vs model langsung",
            "label cocok vs model langsung",
            "label cocok vs Tahap 05 (40 file uji)",
            "selisih probabilitas vs Tahap 05",
            "accuracy pipeline pada 40 file uji",
            "accuracy Tahap 05",
        ],
        "nilai": [
            identity_summary["n_files_checked"],
            f"{max_feature_rel_diff:.3e}",
            f"{max_feature_abs_diff:.3e}",
            f"{max_proba_diff_direct:.3e}",
            f"{n_label_match_176} / 176",
            f"{n_test_label_match} / 40",
            f"{max_test_proba_diff:.3e}",
            f"{test_accuracy:.4f}",
            f"{metrics['accuracy']:.4f}",
        ],
        "acuan": [
            "seluruh data_final/",
            f"toleransi {FEATURE_REL_TOL:g}",
            "kuantisasi PCM16 ulang",
            "toleransi 1e-12",
            "harus 176 / 176",
            "harus 40 / 40",
            f"toleransi {PROBA_MATCH_TOL:g}",
            "harus sama",
            "dari metrics.json",
        ],
    }
)
display(identity_view)
display(
    identity_check.sort_values("feature_max_rel_diff", ascending=False).head(5)[
        [
            "file",
            "split",
            "feature_max_rel_diff",
            "feature_worst",
            "proba_pipeline_vs_recorded",
            "pred_pipeline",
            "pred_recorded",
        ]
    ]
)
print(
    f"Identitas prapemrosesan: 176/176 label cocok, "
    f"selisih fitur relatif maks {max_feature_rel_diff:.3e}, "
    f"selisih probabilitas vs Tahap 05 maks {max_test_proba_diff:.3e}."
)


,pemeriksaan,nilai,acuan
0,file diperiksa,176,seluruh data_final/
1,selisih fitur relatif maksimum,6.105e-05,toleransi 0.001
2,selisih fitur absolut maksimum,2.334e-05,kuantisasi PCM16 ulang
3,selisih probabilitas vs model langsung,2.399e-05,toleransi 1e-12
4,label cocok vs model langsung,176 / 176,harus 176 / 176
5,label cocok vs Tahap 05 (40 file uji),40 / 40,harus 40 / 40
6,selisih probabilitas vs Tahap 05,2.342e-05,toleransi 0.005
7,accuracy pipeline pada 40 file uji,0.7750,harus sama
8,accuracy Tahap 05,0.7750,dari metrics.json


,file,split,feature_max_rel_diff,feature_worst,proba_pipeline_vs_recorded,pred_pipeline,pred_recorded
8,24-006_non-berisik_1.wav,train,0.000061,mse_energy,NaN,0,-1
113,24-131_non-berisik_3.wav,test,0.000061,mse_energy,0.000004,0,0
67,24-087_non-berisik_2.wav,train,0.000061,mse_energy,NaN,0,-1
66,24-087_non-berisik_1.wav,train,0.000061,mse_energy,NaN,0,-1
136,24-168_berisik_1.wav,test,0.000061,mse_energy,0.000001,0,0


Identitas prapemrosesan: 176/176 label cocok, selisih fitur relatif maks 6.105e-05, selisih probabilitas vs Tahap 05 maks 2.342e-05.


## 5. Demo nyata pada file dataset

Demo memakai file asli, bukan data sintetis:

1. **Satu file berisik dan satu file non-berisik** — dipilih deterministik dari 40 file uji (nama terurut per kelas), dijalankan lewat `predict_audio` pada **file MP3 asli di `data_mp3/`**, sehingga seluruh rantai dekode → durasi → normalisasi → fitur → model berjalan dari awal. Label asli dan hasil prediksi dilaporkan apa adanya, termasuk bila salah.
2. **Uji cabang durasi** — file 10 detik (potongan) untuk cabang **pad** dan file 60 detik (gabungan dua rekaman) untuk cabang **crop**, dibuat sementara di `artifacts/06_deployment/demo_audio/`, memastikan aturan durasi bekerja persis seperti kontrak.

Hasil seluruh demo disimpan di `demo_results.csv` dan ditampilkan lengkap dengan probabilitas, aksi durasi, dan status kecocokan.


In [7]:
test_features = features.loc[features["file"].isin(test_predictions["file"])]
demo_picks = pd.concat(
    [
        test_features.loc[test_features["label"] == 0].sort_values("file").head(1),
        test_features.loc[test_features["label"] == 1].sort_values("file").head(1),
    ]
).sort_values("label")

manifest_by_file = transformation_manifest.set_index("file")
demo_rows = []

for row in demo_picks.itertuples():
    mp3_path = DATA_MP3_DIR / manifest_by_file.at[row.file, "mp3_relative_path"]
    result = predict_audio(mp3_path)
    demo_rows.append(
        {
            "skenario": "demo_utama",
            "file": mp3_path.name,
            "path": str(mp3_path),
            "kelas_asli": LABEL_NAME_MAP[int(row.label)],
            "prediksi": result["label_name"],
            "proba_berisik": result["proba_berisik"],
            "cocok": bool(result["label"] == int(row.label)),
            "duration_action": result["duration_action"],
            "durasi_asli_detik": result["original_duration_16k_s"],
            "peak_sebelum_normalisasi": result["peak_before_norm"],
        }
    )

base_row = demo_picks.iloc[0]
other_row = demo_picks.iloc[1]
base_path = DATA_FINAL_DIR / manifest_by_file.at[base_row["file"], "final_relative_path"]
other_path = DATA_FINAL_DIR / manifest_by_file.at[other_row["file"], "final_relative_path"]
base_y, _ = sf.read(base_path, dtype="float32")
other_y, _ = sf.read(other_path, dtype="float32")

short_path = DEMO_AUDIO_DIR / "demo_10_detik.wav"
long_path = DEMO_AUDIO_DIR / "demo_60_detik.wav"
sf.write(short_path, base_y[: int(10 * SR)], SR, subtype="PCM_16")
sf.write(long_path, np.concatenate([base_y, other_y]), SR, subtype="PCM_16")

for scenario, temp_path, reference_label, expected_action in [
    ("cabang_pad_10_detik", short_path, int(base_row["label"]), "pad"),
    ("cabang_crop_60_detik", long_path, int(base_row["label"]), "crop"),
]:
    result = predict_audio(temp_path)
    assert result["duration_action"] == expected_action, (
        f"Aksi durasi {scenario} = {result['duration_action']}, bukan {expected_action}."
    )
    demo_rows.append(
        {
            "skenario": scenario,
            "file": temp_path.name,
            "path": str(temp_path),
            "kelas_asli": LABEL_NAME_MAP[reference_label],
            "prediksi": result["label_name"],
            "proba_berisik": result["proba_berisik"],
            "cocok": bool(result["label"] == reference_label),
            "duration_action": result["duration_action"],
            "durasi_asli_detik": result["original_duration_16k_s"],
            "peak_sebelum_normalisasi": result["peak_before_norm"],
        }
    )

demo_results = pd.DataFrame(demo_rows)
assert len(demo_results) == 4
assert set(demo_results.loc[demo_results["skenario"] == "demo_utama", "kelas_asli"]) == set(
    LABEL_ORDER
)
demo_results.to_csv(DEMO_RESULTS_PATH, index=False)

display(demo_results)
main_demos = demo_results.loc[demo_results["skenario"] == "demo_utama"]
for row in main_demos.itertuples():
    print(
        f"{row.file}: asli {row.kelas_asli} -> prediksi {row.prediksi} "
        f"(p berisik {row.proba_berisik:.4f}), {'cocok' if row.cocok else 'SALAH'}"
    )
print(
    "Cabang durasi teruji: "
    + ", ".join(
        f"{row.skenario} ({row.duration_action}, {row.durasi_asli_detik:.1f} detik)"
        for row in demo_results.loc[demo_results["skenario"] != "demo_utama"].itertuples()
    )
)


,skenario,file,path,kelas_asli,prediksi,proba_berisik,cocok,duration_action,durasi_asli_detik,peak_sebelum_normalisasi
0,demo_utama,24-068_non-berisik_1.mp3,C:\Users\ASUS\Downloads\dataset_30_detik\data_...,non-berisik,berisik,0.740319,False,none,30.0,0.292501
1,demo_utama,24-068_berisik_1.mp3,C:\Users\ASUS\Downloads\dataset_30_detik\data_...,berisik,berisik,0.911437,True,none,30.0,0.809704
2,cabang_pad_10_detik,demo_10_detik.wav,C:\Users\ASUS\Downloads\dataset_30_detik\artif...,non-berisik,non-berisik,0.051632,True,pad,10.0,0.962891
3,cabang_crop_60_detik,demo_60_detik.wav,C:\Users\ASUS\Downloads\dataset_30_detik\artif...,non-berisik,berisik,0.856820,False,crop,60.0,0.999969


24-068_non-berisik_1.mp3: asli non-berisik -> prediksi berisik (p berisik 0.7403), SALAH
24-068_berisik_1.mp3: asli berisik -> prediksi berisik (p berisik 0.9114), cocok
Cabang durasi teruji: cabang_pad_10_detik (pad, 10.0 detik), cabang_crop_60_detik (crop, 60.0 detik)


## 6. Laporan akhir `artifacts/LAPORAN.md`

Dokumen wajib akhir proyek adalah `artifacts/LAPORAN.md`. Isinya mengikuti ketentuan Tahap 06:

1. **Data aktual** — 176 file final, 34 NIM, kelas 88:88, 24 file karantina, format WAV PCM_16 mono 16 kHz × 480.000 sampel, ditulis dari ringkasan Tahap 02–03.
2. **Fitur** — 12 fitur akustik, parameter framing/FFT, dan 8 fitur yang benar-benar dipakai model (hasil `SelectKBest` di dalam pipeline).
3. **Model** — Logistic Regression dalam pipeline `scaler -> select -> clf`, parameter terbaik, skor CV GroupKFold, pembagian hold-out, dan sidik jari file.
4. **Metrik nyata** — hasil hold-out Tahap 05 apa adanya, termasuk status **DI BAWAH SYARAT AWAL 0,85** beserta confidence interval bootstrap.
5. **Deployment** — cara memakai `predict_audio`, hasil demo nyata, dan bukti identitas prapemrosesan 176 file.
6. **Lokasi artefak** dan **limitasi/etika** — privasi, kecilnya jumlah NIM, probabilitas tidak terkalibrasi, dan batas penggunaan alat.

Laporan ditulis dari variabel hasil komputasi (bukan angka ketik manual), disimpan ke `artifacts/LAPORAN.md`, dan dirender di notebook.


In [8]:
def md_table(frame, max_rows=None):
    view = frame if max_rows is None else frame.head(max_rows)
    columns = [str(column) for column in view.columns]
    lines = [
        "| " + " | ".join(columns) + " |",
        "|" + "|".join(["---"] * len(columns)) + "|",
    ]
    for row in view.itertuples(index=False):
        values = []
        for value in row:
            if isinstance(value, (float, np.floating)):
                text = f"{value:.6g}"
            elif isinstance(value, (bool, np.bool_)):
                text = str(bool(value))
            else:
                text = str(value)
            values.append(text.replace("|", "\\|").replace("\n", " "))
        lines.append("| " + " | ".join(values) + " |")
    if len(frame) > len(view):
        lines.append("")
        lines.append(f"*Menampilkan {len(view)} dari {len(frame)} baris.*")
    return "\n".join(lines)


metrics_by_nim = pd.read_csv(ARTIFACT_ROOT / "test_results" / "metrics_by_nim.csv")
worst_nim = metrics_by_nim.loc[metrics_by_nim["accuracy"].idxmin()]
cm = metrics["confusion_matrix"]

data_view = pd.DataFrame(
    {
        "besaran": [
            "File awal (raw_audio)",
            "File final",
            "File karantina",
            "NIM final",
            "Distribusi kelas",
            "NIM gugur",
            "Format final",
            "Durasi",
            "Frame daya nol",
        ],
        "nilai": [
            stage3_summary["n_raw"],
            stage3_summary["n_final"],
            stage3_summary["n_quarantine"],
            stage3_summary["n_nim_final"],
            f"{stage3_summary['class_counts']['0']} non-berisik : {stage3_summary['class_counts']['1']} berisik",
            ", ".join(stage3_summary["dropped_nims"]),
            "WAV PCM_16, mono, 16000 Hz",
            f"{stage3_summary['target_samples']} sampel (30 detik)",
            f"{stage3_summary['n_zero_power_frames']} pada {stage3_summary['n_files_with_zero_power_frames']} file",
        ],
    }
)

feature_view = pd.DataFrame(
    {
        "posisi": ["Input model (12)", "Dipakai classifier (8)"],
        "daftar": [
            ", ".join(FEATURE_COLUMNS),
            ", ".join(selected_from_model),
        ],
    }
)

model_view = pd.DataFrame(
    {
        "komponen": [
            "Algoritma",
            "Pipeline",
            "Parameter terbaik",
            "ROC-AUC CV (GroupKFold 5, grup=NIM)",
            "Baseline dummy",
            "Hold-out",
            "NIM train / test",
            "random_state",
            "Sidik jari file model (sha256)",
            "Versi scikit-learn",
        ],
        "nilai": [
            model_config["model_name"],
            " -> ".join(model_config["pipeline_steps"]),
            str(model_config["best_params"]),
            f"{model_config['best_cv_roc_auc']:.4f}",
            f"{model_config['baseline_cv_roc_auc']:.4f}",
            f"{model_config['holdout']['splitter']} test_size={model_config['holdout']['test_size']}",
            f"{model_config['holdout']['n_nim_train']} NIM ({model_config['holdout']['n_files_train']} file) / "
            f"{model_config['holdout']['n_nim_test']} NIM ({model_config['holdout']['n_files_test']} file)",
            model_config["random_state"],
            model_file_sha256,
            model_config["library_versions"]["scikit-learn"],
        ],
    }
)

eval_view = pd.DataFrame(
    {
        "metrik": [
            "ROC-AUC",
            "CI95 bootstrap",
            "PR-AUC",
            "accuracy",
            "precision (berisik)",
            "recall (berisik)",
            "F1 (berisik)",
            "Confusion matrix",
            "File salah prediksi",
            "Status syarat ROC-AUC >= 0,85",
        ],
        "nilai": [
            f"{metrics['roc_auc']:.4f}",
            f"[{metrics['roc_auc_ci_low']:.4f}, {metrics['roc_auc_ci_high']:.4f}]",
            f"{metrics['pr_auc']:.4f}",
            f"{metrics['accuracy']:.4f}",
            f"{metrics['precision_class_1']:.4f}",
            f"{metrics['recall_class_1']:.4f}",
            f"{metrics['f1_class_1']:.4f}",
            f"TN {cm['tn']} / FP {cm['fp']} / FN {cm['fn']} / TP {cm['tp']}",
            f"{metrics['n_files_wrong']} dari {metrics['n_files_test']}",
            metrics["kelayakan_status"],
        ],
        "acuan": [
            "baseline 0,5000",
            f"{metrics['n_bootstrap']} pengambilan ulang, random_state={metrics['random_state']}",
            f"baseline prevalensi {int(metrics['class_counts_test']['1']) / metrics['n_files_test']:.4f}",
            "40 file uji",
            "kelas positif = berisik",
            "kelas positif = berisik",
            "precision dan recall",
            "threshold 0,50",
            f"terbanyak di NIM {worst_nim['nim']} ({int(worst_nim['n_salah'])} kesalahan)",
            "syarat kelayakan awal",
        ],
    }
)

identity_view = pd.DataFrame(
    {
        "pemeriksaan": [
            "File diperiksa ulang dengan predict_audio",
            "Label cocok vs pipeline langsung",
            "Selisih fitur relatif maksimum",
            "Label cocok vs Tahap 05 (40 file uji)",
            "Selisih probabilitas vs Tahap 05",
            "Accuracy 40 file uji",
        ],
        "nilai": [
            identity_summary["n_files_checked"],
            f"{identity_summary['label_match_vs_direct_176']} / 176",
            f"{identity_summary['max_feature_rel_diff']:.3e}",
            f"{identity_summary['test_label_match_vs_tahap05']} / 40",
            f"{identity_summary['test_max_proba_diff_vs_tahap05']:.3e}",
            f"{identity_summary['test_accuracy_pipeline']:.4f}",
        ],
    }
)

demo_view = demo_results[
    [
        "skenario",
        "file",
        "kelas_asli",
        "prediksi",
        "proba_berisik",
        "cocok",
        "duration_action",
        "durasi_asli_detik",
    ]
]

usage_snippet = """import joblib
from pathlib import Path

# definisi predict_audio ada di notebooks/06_deployment/08_predict_audio.py
model = joblib.load("artifacts/best_model.joblib")
result = predict_audio("data_final/24-068_berisik_1.wav")
print(result["label_name"], result["proba_berisik"])"""

laporan = f"""# LAPORAN Akhir — Klasifikasi Audio Berisik vs Non-Berisik

**Proyek:** Proyek Sains Data A (PSD-A), Universitas Trunojoyo Madura
**Pendekatan:** CRISP-DM (Business Understanding, Data Understanding, Data Preparation, Modeling, Evaluation, Deployment)
**Tanggal laporan:** {pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M UTC")}
**Model:** {model_config["model_name"]} — pipeline `{" -> ".join(model_config["pipeline_steps"])}`

## 1. Ringkasan eksekutif

Model membedakan rekaman kelas **berisik (1)** vs **non-berisik (0)** dari 12 fitur akustik pada audio 30 detik, dan diserahkan sebagai fungsi `predict_audio(path)`.

- **ROC-AUC hold-out = {metrics['roc_auc']:.4f}** (CI95 bootstrap [{metrics['roc_auc_ci_low']:.4f}, {metrics['roc_auc_ci_high']:.4f}], baseline 0,50).
- **Status syarat kelayakan awal ROC-AUC >= 0,85: {metrics['kelayakan_status']}** — dilaporkan apa adanya, tidak disesuaikan.
- Accuracy {metrics['accuracy']:.4f}, precision {metrics['precision_class_1']:.4f}, recall {metrics['recall_class_1']:.4f}, F1 {metrics['f1_class_1']:.4f} pada 40 file uji (7 NIM hold-out); {metrics['n_files_wrong']} file salah prediksi.
- Deployment: prapemrosesan identik Tahap 03 terbukti untuk **{identity_summary['n_files_checked']} file**; demo nyata berjalan pada file MP3 asli.

## 2. Data aktual

{md_table(data_view)}

Catatan: 24 file karantina adalah duplikat hasil seleksi hash, tetap disimpan untuk audit; `raw_audio/` berisi 200 file dan tidak diubah.

## 3. Fitur

{md_table(feature_view)}

Parameter ekstraksi: sr={SR} Hz, frame_length={FRAME_LENGTH}, hop_length={HOP_LENGTH}, n_fft={N_FFT}, window Hann simetris, tanpa `center`, rolloff {ROLLOFF_PERCENT}, epsilon `{EPS:.3e}`; kontrak lengkap di `artifacts/03_data_preparation/preprocessing_contract.json`. Scaling dan seleksi fitur dipelajari **di dalam pipeline** pada data latih, bukan di muka data.

## 4. Model dan validasi (Tahap 04)

{md_table(model_view)}

## 5. Evaluasi hold-out (Tahap 05)

Diuji **sekali** pada 7 NIM test yang dikunci sejak Tahap 04; model tidak di-fit ulang.

{md_table(eval_view)}

Rincian per NIM test:

{md_table(metrics_by_nim)}

## 6. Deployment

Pemakaian:

```python
{usage_snippet}
```

Fungsi `predict_audio` melakukan: dekode `librosa.load(sr=16000, mono=True)` -> crop/pad tengah ke {TARGET_SAMPLES} sampel -> peak normalization -> 12 fitur -> pipeline tersimpan -> label dan probabilitas (threshold {TEST_THRESHOLD}).

### Demo nyata

{md_table(demo_view)}

### Bukti identitas prapemrosesan

{md_table(identity_view)}

## 7. Lokasi artefak

- `artifacts/best_model.joblib` — model pipeline siap pakai.
- `artifacts/features.csv` — 176 baris fitur Tahap 03.
- `artifacts/02_data_understanding/`, `artifacts/03_data_preparation/` — laporan data dan manifest.
- `artifacts/04_modeling/` — konfigurasi model, split hold-out, laporan modeling.
- `artifacts/test_results/` — prediksi, metrik, confusion, per-NIM, kesalahan, dan 5 gambar evaluasi.
- `artifacts/05_evaluation/` — laporan evaluasi dan validasi.
- `artifacts/06_deployment/` — bukti identitas, hasil demo, laporan, dan validasi tahap ini.
- `notebooks/06_deployment/` — sumber cell notebook deployment.

## 8. Limitasi dan etika

1. **Kelayakan:** ROC-AUC {metrics['roc_auc']:.4f} **di bawah syarat awal 0,85**, dan CI95 [{metrics['roc_auc_ci_low']:.4f}, {metrics['roc_auc_ci_high']:.4f}] memuat 0,85 — kelayakan pada populasi tidak dapat dipastikan dari 40 file ini saja.
2. **Jumlah kecil:** hanya {metrics['n_nim_test']} NIM test / {metrics['n_files_test']} file; {int(worst_nim['n_salah'])} dari {metrics['n_files_wrong']} kesalahan menumpuk pada NIM `{worst_nim['nim']}`.
3. **Probabilitas** adalah skor model, belum terkalibrasi; threshold {TEST_THRESHOLD} tidak dituning.
4. **Label** berasal dari protokol pengumpulan, bukan audit kebenaran mutlak.
5. **Privasi:** rekaman berisi suara orang; gunakan hanya untuk proyek pembelajaran, batasi akses, jangan melakukan pengenalan pembicara.
6. **Bukan penilai disiplin:** alat ini penyaringan awal, bukan pengganti observasi manusia; batas perangkat/ruangan berlaku.

## 9. Kesimpulan

1. Pipeline CRISP-DM lengkap 01-06 berjalan dari 200 file mentah menjadi model teruji dan fungsi prediksi yang terbukti identik prapemrosesannya.
2. Metrik hold-out dilaporkan apa adanya, termasuk status {metrics["kelayakan_status"].lower()} terhadap syarat 0,85.
3. `predict_audio(path)` siap dipakai dengan artefak yang ada; perbaikan selanjutnya (bila diperlukan) harus melalui siklus data/model baru, bukan penalaan pada data uji.
"""

LAPORAN_PATH.write_text(laporan, encoding="utf-8")
assert LAPORAN_PATH.is_file() and LAPORAN_PATH.stat().st_size > 3000

print(f"Laporan akhir: {LAPORAN_PATH} ({LAPORAN_PATH.stat().st_size} bytes)")
display(Markdown(laporan))


Laporan akhir: C:\Users\ASUS\Downloads\dataset_30_detik\artifacts\LAPORAN.md (7326 bytes)


# LAPORAN Akhir — Klasifikasi Audio Berisik vs Non-Berisik

**Proyek:** Proyek Sains Data A (PSD-A), Universitas Trunojoyo Madura
**Pendekatan:** CRISP-DM (Business Understanding, Data Understanding, Data Preparation, Modeling, Evaluation, Deployment)
**Tanggal laporan:** 2026-10-24 02:42 UTC
**Model:** Logistic Regression — pipeline `scaler -> select -> clf`

## 1. Ringkasan eksekutif

Model membedakan rekaman kelas **berisik (1)** vs **non-berisik (0)** dari 12 fitur akustik pada audio 30 detik, dan diserahkan sebagai fungsi `predict_audio(path)`.

- **ROC-AUC hold-out = 0.8296** (CI95 bootstrap [0.6767, 0.9549], baseline 0,50).
- **Status syarat kelayakan awal ROC-AUC >= 0,85: DI BAWAH SYARAT AWAL** — dilaporkan apa adanya, tidak disesuaikan.
- Accuracy 0.7750, precision 0.7500, recall 0.7895, F1 0.7692 pada 40 file uji (7 NIM hold-out); 9 file salah prediksi.
- Deployment: prapemrosesan identik Tahap 03 terbukti untuk **176 file**; demo nyata berjalan pada file MP3 asli.

## 2. Data aktual

| besaran | nilai |
|---|---|
| File awal (raw_audio) | 200 |
| File final | 176 |
| File karantina | 24 |
| NIM final | 34 |
| Distribusi kelas | 88 non-berisik : 88 berisik |
| NIM gugur | 24-123 |
| Format final | WAV PCM_16, mono, 16000 Hz |
| Durasi | 480000 sampel (30 detik) |
| Frame daya nol | 1677 pada 52 file |

Catatan: 24 file karantina adalah duplikat hasil seleksi hash, tetap disimpan untuk audit; `raw_audio/` berisi 200 file dan tidak diubah.

## 3. Fitur

| posisi | daftar |
|---|---|
| Input model (12) | mse_energy, rms_mean, rms_std, rms_max, rms_min, zcr_mean, zcr_std, spectral_centroid_mean, spectral_centroid_std, spectral_flatness_mean, spectral_rolloff_mean, spectral_bandwidth_mean |
| Dipakai classifier (8) | mse_energy, rms_mean, rms_std, rms_max, rms_min, zcr_mean, spectral_centroid_mean, spectral_rolloff_mean |

Parameter ekstraksi: sr=16000 Hz, frame_length=400, hop_length=160, n_fft=512, window Hann simetris, tanpa `center`, rolloff 0.85, epsilon `2.220e-16`; kontrak lengkap di `artifacts/03_data_preparation/preprocessing_contract.json`. Scaling dan seleksi fitur dipelajari **di dalam pipeline** pada data latih, bukan di muka data.

## 4. Model dan validasi (Tahap 04)

| komponen | nilai |
|---|---|
| Algoritma | Logistic Regression |
| Pipeline | scaler -> select -> clf |
| Parameter terbaik | {'clf__C': 10.0, 'select__k': 8} |
| ROC-AUC CV (GroupKFold 5, grup=NIM) | 0.8442 |
| Baseline dummy | 0.5000 |
| Hold-out | GroupShuffleSplit test_size=0.2 |
| NIM train / test | 27 NIM (136 file) / 7 NIM (40 file) |
| random_state | 42 |
| Sidik jari file model (sha256) | 964a1550c9c06123e00599a17546de694e64de19aee833a771c9d6ceaa35390a |
| Versi scikit-learn | 1.8.0 |

## 5. Evaluasi hold-out (Tahap 05)

Diuji **sekali** pada 7 NIM test yang dikunci sejak Tahap 04; model tidak di-fit ulang.

| metrik | nilai | acuan |
|---|---|---|
| ROC-AUC | 0.8296 | baseline 0,5000 |
| CI95 bootstrap | [0.6767, 0.9549] | 2000 pengambilan ulang, random_state=42 |
| PR-AUC | 0.8481 | baseline prevalensi 0.4750 |
| accuracy | 0.7750 | 40 file uji |
| precision (berisik) | 0.7500 | kelas positif = berisik |
| recall (berisik) | 0.7895 | kelas positif = berisik |
| F1 (berisik) | 0.7692 | precision dan recall |
| Confusion matrix | TN 16 / FP 5 / FN 4 / TP 15 | threshold 0,50 |
| File salah prediksi | 9 dari 40 | terbanyak di NIM 24-168 (5 kesalahan) |
| Status syarat ROC-AUC >= 0,85 | DI BAWAH SYARAT AWAL | syarat kelayakan awal |

Rincian per NIM test:

| nim | n_files | n_class_0 | n_class_1 | n_benar | n_pred_1 | n_salah | accuracy |
|---|---|---|---|---|---|---|---|
| 24-068 | 6 | 3 | 3 | 4 | 5 | 2 | 0.6667 |
| 24-100 | 6 | 3 | 3 | 6 | 3 | 0 | 1 |
| 24-119 | 6 | 3 | 3 | 6 | 3 | 0 | 1 |
| 24-131 | 6 | 3 | 3 | 6 | 3 | 0 | 1 |
| 24-145 | 6 | 3 | 3 | 4 | 3 | 2 | 0.6667 |
| 24-168 | 6 | 3 | 3 | 1 | 2 | 5 | 0.1667 |
| 24-169 | 4 | 3 | 1 | 4 | 1 | 0 | 1 |

## 6. Deployment

Pemakaian:

```python
import joblib
from pathlib import Path

# definisi predict_audio ada di notebooks/06_deployment/08_predict_audio.py
model = joblib.load("artifacts/best_model.joblib")
result = predict_audio("data_final/24-068_berisik_1.wav")
print(result["label_name"], result["proba_berisik"])
```

Fungsi `predict_audio` melakukan: dekode `librosa.load(sr=16000, mono=True)` -> crop/pad tengah ke 480000 sampel -> peak normalization -> 12 fitur -> pipeline tersimpan -> label dan probabilitas (threshold 0.5).

### Demo nyata

| skenario | file | kelas_asli | prediksi | proba_berisik | cocok | duration_action | durasi_asli_detik |
|---|---|---|---|---|---|---|---|
| demo_utama | 24-068_non-berisik_1.mp3 | non-berisik | berisik | 0.740319 | False | none | 30 |
| demo_utama | 24-068_berisik_1.mp3 | berisik | berisik | 0.911437 | True | none | 30 |
| cabang_pad_10_detik | demo_10_detik.wav | non-berisik | non-berisik | 0.0516325 | True | pad | 10 |
| cabang_crop_60_detik | demo_60_detik.wav | non-berisik | berisik | 0.85682 | False | crop | 60 |

### Bukti identitas prapemrosesan

| pemeriksaan | nilai |
|---|---|
| File diperiksa ulang dengan predict_audio | 176 |
| Label cocok vs pipeline langsung | 176 / 176 |
| Selisih fitur relatif maksimum | 6.105e-05 |
| Label cocok vs Tahap 05 (40 file uji) | 40 / 40 |
| Selisih probabilitas vs Tahap 05 | 2.342e-05 |
| Accuracy 40 file uji | 0.7750 |

## 7. Lokasi artefak

- `artifacts/best_model.joblib` — model pipeline siap pakai.
- `artifacts/features.csv` — 176 baris fitur Tahap 03.
- `artifacts/02_data_understanding/`, `artifacts/03_data_preparation/` — laporan data dan manifest.
- `artifacts/04_modeling/` — konfigurasi model, split hold-out, laporan modeling.
- `artifacts/test_results/` — prediksi, metrik, confusion, per-NIM, kesalahan, dan 5 gambar evaluasi.
- `artifacts/05_evaluation/` — laporan evaluasi dan validasi.
- `artifacts/06_deployment/` — bukti identitas, hasil demo, laporan, dan validasi tahap ini.
- `notebooks/06_deployment/` — sumber cell notebook deployment.

## 8. Limitasi dan etika

1. **Kelayakan:** ROC-AUC 0.8296 **di bawah syarat awal 0,85**, dan CI95 [0.6767, 0.9549] memuat 0,85 — kelayakan pada populasi tidak dapat dipastikan dari 40 file ini saja.
2. **Jumlah kecil:** hanya 7 NIM test / 40 file; 5 dari 9 kesalahan menumpuk pada NIM `24-168`.
3. **Probabilitas** adalah skor model, belum terkalibrasi; threshold 0.5 tidak dituning.
4. **Label** berasal dari protokol pengumpulan, bukan audit kebenaran mutlak.
5. **Privasi:** rekaman berisi suara orang; gunakan hanya untuk proyek pembelajaran, batasi akses, jangan melakukan pengenalan pembicara.
6. **Bukan penilai disiplin:** alat ini penyaringan awal, bukan pengganti observasi manusia; batas perangkat/ruangan berlaku.

## 9. Kesimpulan

1. Pipeline CRISP-DM lengkap 01-06 berjalan dari 200 file mentah menjadi model teruji dan fungsi prediksi yang terbukti identik prapemrosesannya.
2. Metrik hold-out dilaporkan apa adanya, termasuk status di bawah syarat awal terhadap syarat 0,85.
3. `predict_audio(path)` siap dipakai dengan artefak yang ada; perbaikan selanjutnya (bila diperlukan) harus melalui siklus data/model baru, bukan penalaan pada data uji.


## 7. Validasi akhir Tahap 06

Seluruh klaim deployment diverifikasi dari artefak dan eksekusi nyata, bukan dari asumsi:

1. **Pintu masuk** — validasi Tahap 03, 04, 05 berstatus PASS; struktur pipeline, parameter, 12 fitur, dan kontrak prapemrosesan tidak berubah.
2. **Identitas model** — probabilitas langsung dari `features.csv` identik dengan `test_predictions.csv` (selisih < 1e-12); sidik jari SHA-256 file model tercatat.
3. **Identitas prapemrosesan** — seluruh 176 file menghasilkan fitur relatif selisihnya di bawah toleransi, label konsisten, dan reproduksi 40 file uji cocok dengan Tahap 05.
4. **Demo** — dua kelas terwakili, cabang pad dan crop teruji, file demo tersimpan.
5. **Laporan** — `artifacts/LAPORAN.md` ada, memuat metrik nyata beserta status syarat 0,85, lokasi artefak, dan limitasi.
6. **Kebutuhan tahap lain** — keluaran `artifacts/test_results/` lengkap; `raw_audio/` tetap 200 file; `data_final/` tetap 176 WAV.

Bila satu saja pemeriksaan gagal, tahap dianggap tidak lulus.


In [9]:
expected_test_results = [
    "test_predictions.csv",
    "metrics.json",
    "classification_report.csv",
    "confusion_matrix.csv",
    "metrics_by_nim.csv",
    "errors.csv",
]
expected_figures = [
    "roc_curve.png",
    "pr_curve.png",
    "confusion_matrix.png",
    "proba_distribution.png",
    "roc_auc_bootstrap.png",
]
test_results_ok = all(
    (ARTIFACT_ROOT / "test_results" / name).is_file() for name in expected_test_results
) and all((ARTIFACT_ROOT / "test_results" / "figures" / name).is_file() for name in expected_figures)

deployment_artifacts_ok = all(path.is_file() for path in [
    IDENTITY_CHECK_PATH,
    IDENTITY_SUMMARY_PATH,
    DEMO_RESULTS_PATH,
    LAPORAN_PATH,
])
n_raw_now = sum(1 for path in RAW_DIR.rglob("*") if path.is_file())
n_final_now = sum(1 for path in DATA_FINAL_DIR.rglob("*.wav") if path.is_file())

demo_main = demo_results.loc[demo_results["skenario"] == "demo_utama"]
demo_classes_ok = set(demo_main["kelas_asli"]) == set(LABEL_ORDER)
demo_pad_ok = bool(
    (demo_results["skenario"] == "cabang_pad_10_detik").any()
    and demo_results.loc[
        demo_results["skenario"] == "cabang_pad_10_detik", "duration_action"
    ].iloc[0]
    == "pad"
)
demo_crop_ok = bool(
    (demo_results["skenario"] == "cabang_crop_60_detik").any()
    and demo_results.loc[
        demo_results["skenario"] == "cabang_crop_60_detik", "duration_action"
    ].iloc[0]
    == "crop"
)
demo_files_ok = (DEMO_AUDIO_DIR / "demo_10_detik.wav").is_file() and (
    DEMO_AUDIO_DIR / "demo_60_detik.wav"
).is_file()

laporan_text = LAPORAN_PATH.read_text(encoding="utf-8") if LAPORAN_PATH.is_file() else ""
laporan_size = LAPORAN_PATH.stat().st_size if LAPORAN_PATH.is_file() else 0
roc_text = f"{metrics['roc_auc']:.4f}"

checks = {
    "gate03_pass": stage3_validation["status"] == "PASS",
    "gate04_pass": stage4_validation["status"] == "PASS",
    "gate05_pass": stage5_validation["status"] == "PASS",
    "pipeline_struktur": list(saved_model.named_steps) == ["scaler", "select", "clf"],
    "parameter_tidak_berubah": all(
        saved_model.get_params()[key] == value
        for key, value in model_config["best_params"].items()
    ),
    "fitur_input_12": list(features.columns)[4:] == FEATURE_COLUMNS
    and int(saved_model.n_features_in_) == 12,
    "fitur_terpilih_8": selected_from_model == model_config["selected_features_final"]
    and len(selected_from_model) == 8,
    "kontrak_prapemrosesan_cocok": contract["sample_rate"] == SR
    and contract["target_samples"] == TARGET_SAMPLES
    and contract["frame_length"] == FRAME_LENGTH
    and contract["hop_length"] == HOP_LENGTH
    and contract["n_fft"] == N_FFT,
    "sidik_jari_model_tercatat": len(model_file_sha256) == 64 and len(model_fingerprint) == 64,
    "proba_model_identik": model_identity_max_diff < MODEL_IDENTITY_TOL,
    "predict_audio_terdefinisi": callable(globals().get("predict_audio")),
    "fitur_predict_audio_sah": list(smoke_result["features"]) == FEATURE_COLUMNS
    and all(np.isfinite(smoke_result["features"][c]) for c in FEATURE_COLUMNS),
    "identitas_fitur_176": identity_summary["n_files_checked"] == 176
    and identity_summary["max_feature_rel_diff"] < FEATURE_REL_TOL,
    "label_konsisten_176": identity_summary["label_match_vs_direct_176"] == 176,
    "reproduksi_test_40": identity_summary["test_label_match_vs_tahap05"] == 40,
    "probabilitas_test_konsisten": identity_summary["test_max_proba_diff_vs_tahap05"]
    <= PROBA_MATCH_TOL,
    "accuracy_sama_tahap05": abs(
        identity_summary["test_accuracy_pipeline"] - identity_summary["test_accuracy_tahap05"]
    )
    < 1e-12,
    "demo_duakelas": demo_classes_ok,
    "demo_empat_skenario": len(demo_results) == 4,
    "cabang_pad_teruji": demo_pad_ok,
    "cabang_crop_teruji": demo_crop_ok,
    "file_demo_tersimpan": demo_files_ok,
    "laporan_ada": laporan_size > 3000,
    "laporan_memuat_metrik": roc_text in laporan_text
    and metrics["kelayakan_status"] in laporan_text,
    "laporan_memuat_limitasi": "Limitasi dan etika" in laporan_text,
    "laporan_memuat_artefak": "artifacts/best_model.joblib" in laporan_text,
    "test_results_lengkap": test_results_ok,
    "artefak_06_lengkap": deployment_artifacts_ok,
    "raw_audio_tetap_200": n_raw_now == 200,
    "data_final_tetap_176": n_final_now == 176,
    "threshold_konsisten": TEST_THRESHOLD == 0.5
    and float(metrics["test_threshold"]) == TEST_THRESHOLD,
}

check_table = pd.DataFrame(
    {
        "pemeriksaan": list(checks),
        "lulus": [bool(value) for value in checks.values()],
    }
)
check_table["status"] = check_table["lulus"].map({True: "PASS", False: "FAIL"})
display(check_table)

if not check_table["lulus"].all():
    failed = check_table.loc[~check_table["lulus"], "pemeriksaan"].tolist()
    raise ValueError(f"Validasi akhir Tahap 06 gagal: {failed}")

checks_passed = int(check_table["lulus"].sum())
validation_summary = {
    "status": "PASS",
    "model_name": model_config["model_name"],
    "model_class": model_config["model_class"],
    "model_file_sha256": model_file_sha256,
    "model_unchanged": bool(model_identity_max_diff < MODEL_IDENTITY_TOL),
    "predict_audio_defined": True,
    "n_files_identity_checked": int(identity_summary["n_files_checked"]),
    "max_feature_rel_diff": identity_summary["max_feature_rel_diff"],
    "test_label_match_vs_tahap05": identity_summary["test_label_match_vs_tahap05"],
    "test_max_proba_diff_vs_tahap05": identity_summary["test_max_proba_diff_vs_tahap05"],
    "demo_main_files": demo_main["file"].tolist(),
    "demo_classes": sorted(demo_main["kelas_asli"].tolist()),
    "laporan_bytes": int(laporan_size),
    "roc_auc_tahap05": metrics["roc_auc"],
    "kelayakan_status": metrics["kelayakan_status"],
    "checks_passed": checks_passed,
    "checks_total": int(len(check_table)),
}
ARTIFACT_DIR.joinpath("validation_checks.csv").write_text(
    check_table.to_csv(index=False), encoding="utf-8"
)
ARTIFACT_DIR.joinpath("validation_summary.json").write_text(
    json.dumps(validation_summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

report_lines = [
    "# Laporan Validasi Tahap 06",
    "",
    f"Status: **PASS**",
    "",
    "## Angka aktual",
    f"- Model: {model_config['model_name']} (pipeline {' -> '.join(model_config['pipeline_steps'])})",
    f"- Sidik jari file model: `{model_file_sha256}`",
    f"- File diperiksa identitas prapemrosesan: {identity_summary['n_files_checked']}",
    f"- Selisih fitur relatif maksimum: {identity_summary['max_feature_rel_diff']:.3e}",
    f"- Label cocok vs Tahap 05 (40 file uji): {identity_summary['test_label_match_vs_tahap05']}/40",
    f"- Selisih probabilitas vs Tahap 05: {identity_summary['test_max_proba_diff_vs_tahap05']:.3e}",
    f"- Demo: {', '.join(demo_main['file'])} (kelas {', '.join(sorted(demo_main['kelas_asli']))})",
    f"- Cabang durasi teruji: pad (10 detik), crop (60 detik)",
    f"- `artifacts/LAPORAN.md`: {laporan_size} bytes",
    f"- ROC-AUC Tahap 05 yang dilaporkan: {metrics['roc_auc']:.4f} — {metrics['kelayakan_status']}",
    f"- Pemeriksaan lulus: {checks_passed}/{len(check_table)}",
    "",
    "## Catatan",
    "- Tidak ada pelatihan, penalaan, atau pemilihan fitur pada tahap ini.",
    "- Fungsi `predict_audio` memakai kontrak prapemrosesan Tahap 03 tanpa perubahan.",
    "- Metrik hold-out dilaporkan apa adanya, termasuk status di bawah syarat 0,85.",
    "",
]
ARTIFACT_DIR.joinpath("validation_report.md").write_text(
    "\n".join(report_lines), encoding="utf-8"
)

print(f"Validasi akhir Tahap 06: PASS ({checks_passed}/{len(check_table)})")
print(f"Laporan: {ARTIFACT_DIR / 'validation_report.md'}")


,pemeriksaan,lulus,status
0,gate03_pass,True,PASS
1,gate04_pass,True,PASS
2,gate05_pass,True,PASS
3,pipeline_struktur,True,PASS
4,parameter_tidak_berubah,True,PASS
5,fitur_input_12,True,PASS
6,fitur_terpilih_8,True,PASS
7,kontrak_prapemrosesan_cocok,True,PASS
8,sidik_jari_model_tercatat,True,PASS
9,proba_model_identik,True,PASS


Validasi akhir Tahap 06: PASS (31/31)
Laporan: C:\Users\ASUS\Downloads\dataset_30_detik\artifacts\06_deployment\validation_report.md


In [10]:
report = f"""# 06 — Deployment: Laporan Aktual

## Ringkasan

Fungsi **`predict_audio(path)`** dibangun memakai `artifacts/best_model.joblib` **tanpa training ulang**, dengan prapemrosesan identik Tahap 03, dan didemokan pada file asli dataset.

- Model: **{model_config['model_name']}** (pipeline `{' -> '.join(model_config['pipeline_steps'])}`), parameter {model_config['best_params']}, sidik jari file `{model_file_sha256[:16]}...`.
- Identitas model: selisih probabilitas vs `test_predictions.csv` Tahap 05 **{model_identity_max_diff:.3e}** (toleransi 1e-12) — model tidak diganti, tidak di-fit ulang.
- Identitas prapemrosesan: **{identity_summary['n_files_checked']}/176 file** diperiksa; selisih fitur relatif maksimum **{identity_summary['max_feature_rel_diff']:.3e}** (toleransi {FEATURE_REL_TOL:g}), label cocok {identity_summary['label_match_vs_direct_176']}/176.
- Reproduksi Tahap 05: 40/40 label cocok, selisih probabilitas maksimum {identity_summary['test_max_proba_diff_vs_tahap05']:.3e} (toleransi {PROBA_MATCH_TOL:g}); accuracy 40 file uji {identity_summary['test_accuracy_pipeline']:.4f} = accuracy Tahap 05.
- Demo: 2 file utama (satu per kelas) + 2 cabang durasi (pad 10 detik, crop 60 detik), seluruhnya berjalan nyata.
- Laporan akhir: **`artifacts/LAPORAN.md`** ({laporan_size} bytes).
- Pemeriksaan validasi: {check_table['lulus'].sum()}/{len(check_table)} PASS.

## Yang dibangun

Fungsi `predict_audio(path)` (cell `08_predict_audio.py`):

1. `librosa.load(path, sr=16000, mono=True)`;
2. crop/pad tengah ke {TARGET_SAMPLES} sampel;
3. peak normalization `y / max(|y|)`, hasil `float32`;
4. 12 fitur dengan framing {FRAME_LENGTH}/{HOP_LENGTH}, Hann simetris, rFFT {N_FFT};
5. pipeline tersimpan (scaler + seleksi fitur + klasifikasi) -> label pada threshold {TEST_THRESHOLD} beserta probabilitas kedua kelas.

Fungsi menolak file tidak ada dan sinyal peak nol; seluruh konstanta cocok dengan `preprocessing_contract.json`.

## Demo nyata

{md_table(demo_view)}

- Dua file utama diambil deterministik dari 40 file uji dan dijalankan lewat **file MP3 asli** (`data_mp3/`), bukan salinan hasil.
- Hasil dilaporkan apa adanya: file yang keliru tetap ditulis sebagai keliru.
- Cabang `pad` (10 detik) dan `crop` (60 detik) teruji sesuai kontrak durasi.

## Bukti identitas prapemrosesan

{md_table(identity_view)}

Sumber perbedaan yang tersisa hanyalah kuantisasi ulang PCM16 ketika file final dinormalisasi kembali (faktor ~3e-5) — jauh di bawah toleransi dan tidak mengubah satu pun label uji.

## Laporan akhir

`artifacts/LAPORAN.md` memuat: data aktual Tahap 02-03, 12 fitur dan 8 fitur terpilih, model dan konfigurasi Tahap 04, metrik hold-out Tahap 05 **apa adanya** (ROC-AUC {metrics['roc_auc']:.4f}, status {metrics['kelayakan_status']}), cara pemakaian `predict_audio`, lokasi seluruh artefak, serta limitasi dan etika.

## Validasi akhir

Semua pemeriksaan lulus: **{check_table['lulus'].sum()}/{len(check_table)}**.

{md_table(check_table[['pemeriksaan', 'status']])}

## Batasan

- Probabilitas adalah skor model, bukan keyakinan terkalibrasi; threshold {TEST_THRESHOLD} tidak dituning.
- ROC-AUC {metrics['roc_auc']:.4f} tetap **di bawah syarat awal 0,85**; CI95 [{metrics['roc_auc_ci_low']:.4f}, {metrics['roc_auc_ci_high']:.4f}] memuat 0,85.
- Evaluasi hanya {metrics['n_nim_test']} NIM / {metrics['n_files_test']} file; kesalahan menumpuk pada NIM {worst_nim['nim']}.
- Fungsi mengasumsikan audio berdurasi mendekati 30 detik; durasi ekstrem dikenali lewat crop/pad tetapi kualitas prediksi menurun bila bahan utama terbuang.
- Rekaman berisi suara orang: gunakan hanya untuk proyek pembelajaran; tanpa pengenalan pembicara.

## Kesimpulan dan arah lanjut

1. Deployment selesai: `predict_audio(path)` terbukti identik prapemrosesannya untuk 176 file dan mereproduksi evaluasi Tahap 05 untuk 40 file uji.
2. `artifacts/LAPORAN.md` berisi seluruh angka aktual, lokasi artefak, dan limitasi untuk konsumen laporan.
3. Bila kelayakan 0,85 ingin dipenuhi, langkah berikutnya adalah siklus data/model baru (data lebih banyak dan lebih beragam), bukan penilaian ulang pada hold-out yang sama.

**Deployment selesai dan tervalidasi. Tanpa training ulang; metrik dilaporkan apa adanya.**
"""

ARTIFACT_DIR.joinpath("deployment_report.md").write_text(report, encoding="utf-8")

deployment_summary = {
    "status": "PASS",
    "model_name": model_config["model_name"],
    "model_class": model_config["model_class"],
    "model_file_sha256": model_file_sha256,
    "model_identity_max_diff": model_identity_max_diff,
    "predict_audio_callable": True,
    "test_threshold": TEST_THRESHOLD,
    "identity": identity_summary,
    "demo_main": demo_main[["file", "kelas_asli", "prediksi", "proba_berisik", "cocok"]].to_dict(
        orient="records"
    ),
    "demo_duration_actions": {
        row.skenario: row.duration_action for row in demo_results.itertuples()
    },
    "laporan_bytes": int(laporan_size),
    "roc_auc_tahap05": metrics["roc_auc"],
    "kelayakan_status": metrics["kelayakan_status"],
    "checks_passed": int(check_table["lulus"].sum()),
    "checks_total": int(len(check_table)),
    "random_state": RANDOM_STATE,
}
ARTIFACT_DIR.joinpath("deployment_summary.json").write_text(
    json.dumps(deployment_summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"Laporan tahap: {ARTIFACT_DIR / 'deployment_report.md'}")
display(Markdown(report))


Laporan tahap: C:\Users\ASUS\Downloads\dataset_30_detik\artifacts\06_deployment\deployment_report.md


# 06 — Deployment: Laporan Aktual

## Ringkasan

Fungsi **`predict_audio(path)`** dibangun memakai `artifacts/best_model.joblib` **tanpa training ulang**, dengan prapemrosesan identik Tahap 03, dan didemokan pada file asli dataset.

- Model: **Logistic Regression** (pipeline `scaler -> select -> clf`), parameter {'clf__C': 10.0, 'select__k': 8}, sidik jari file `964a1550c9c06123...`.
- Identitas model: selisih probabilitas vs `test_predictions.csv` Tahap 05 **1.110e-16** (toleransi 1e-12) — model tidak diganti, tidak di-fit ulang.
- Identitas prapemrosesan: **176/176 file** diperiksa; selisih fitur relatif maksimum **6.105e-05** (toleransi 0.001), label cocok 176/176.
- Reproduksi Tahap 05: 40/40 label cocok, selisih probabilitas maksimum 2.342e-05 (toleransi 0.005); accuracy 40 file uji 0.7750 = accuracy Tahap 05.
- Demo: 2 file utama (satu per kelas) + 2 cabang durasi (pad 10 detik, crop 60 detik), seluruhnya berjalan nyata.
- Laporan akhir: **`artifacts/LAPORAN.md`** (7326 bytes).
- Pemeriksaan validasi: 31/31 PASS.

## Yang dibangun

Fungsi `predict_audio(path)` (cell `08_predict_audio.py`):

1. `librosa.load(path, sr=16000, mono=True)`;
2. crop/pad tengah ke 480000 sampel;
3. peak normalization `y / max(|y|)`, hasil `float32`;
4. 12 fitur dengan framing 400/160, Hann simetris, rFFT 512;
5. pipeline tersimpan (scaler + seleksi fitur + klasifikasi) -> label pada threshold 0.5 beserta probabilitas kedua kelas.

Fungsi menolak file tidak ada dan sinyal peak nol; seluruh konstanta cocok dengan `preprocessing_contract.json`.

## Demo nyata

| skenario | file | kelas_asli | prediksi | proba_berisik | cocok | duration_action | durasi_asli_detik |
|---|---|---|---|---|---|---|---|
| demo_utama | 24-068_non-berisik_1.mp3 | non-berisik | berisik | 0.740319 | False | none | 30 |
| demo_utama | 24-068_berisik_1.mp3 | berisik | berisik | 0.911437 | True | none | 30 |
| cabang_pad_10_detik | demo_10_detik.wav | non-berisik | non-berisik | 0.0516325 | True | pad | 10 |
| cabang_crop_60_detik | demo_60_detik.wav | non-berisik | berisik | 0.85682 | False | crop | 60 |

- Dua file utama diambil deterministik dari 40 file uji dan dijalankan lewat **file MP3 asli** (`data_mp3/`), bukan salinan hasil.
- Hasil dilaporkan apa adanya: file yang keliru tetap ditulis sebagai keliru.
- Cabang `pad` (10 detik) dan `crop` (60 detik) teruji sesuai kontrak durasi.

## Bukti identitas prapemrosesan

| pemeriksaan | nilai |
|---|---|
| File diperiksa ulang dengan predict_audio | 176 |
| Label cocok vs pipeline langsung | 176 / 176 |
| Selisih fitur relatif maksimum | 6.105e-05 |
| Label cocok vs Tahap 05 (40 file uji) | 40 / 40 |
| Selisih probabilitas vs Tahap 05 | 2.342e-05 |
| Accuracy 40 file uji | 0.7750 |

Sumber perbedaan yang tersisa hanyalah kuantisasi ulang PCM16 ketika file final dinormalisasi kembali (faktor ~3e-5) — jauh di bawah toleransi dan tidak mengubah satu pun label uji.

## Laporan akhir

`artifacts/LAPORAN.md` memuat: data aktual Tahap 02-03, 12 fitur dan 8 fitur terpilih, model dan konfigurasi Tahap 04, metrik hold-out Tahap 05 **apa adanya** (ROC-AUC 0.8296, status DI BAWAH SYARAT AWAL), cara pemakaian `predict_audio`, lokasi seluruh artefak, serta limitasi dan etika.

## Validasi akhir

Semua pemeriksaan lulus: **31/31**.

| pemeriksaan | status |
|---|---|
| gate03_pass | PASS |
| gate04_pass | PASS |
| gate05_pass | PASS |
| pipeline_struktur | PASS |
| parameter_tidak_berubah | PASS |
| fitur_input_12 | PASS |
| fitur_terpilih_8 | PASS |
| kontrak_prapemrosesan_cocok | PASS |
| sidik_jari_model_tercatat | PASS |
| proba_model_identik | PASS |
| predict_audio_terdefinisi | PASS |
| fitur_predict_audio_sah | PASS |
| identitas_fitur_176 | PASS |
| label_konsisten_176 | PASS |
| reproduksi_test_40 | PASS |
| probabilitas_test_konsisten | PASS |
| accuracy_sama_tahap05 | PASS |
| demo_duakelas | PASS |
| demo_empat_skenario | PASS |
| cabang_pad_teruji | PASS |
| cabang_crop_teruji | PASS |
| file_demo_tersimpan | PASS |
| laporan_ada | PASS |
| laporan_memuat_metrik | PASS |
| laporan_memuat_limitasi | PASS |
| laporan_memuat_artefak | PASS |
| test_results_lengkap | PASS |
| artefak_06_lengkap | PASS |
| raw_audio_tetap_200 | PASS |
| data_final_tetap_176 | PASS |
| threshold_konsisten | PASS |

## Batasan

- Probabilitas adalah skor model, bukan keyakinan terkalibrasi; threshold 0.5 tidak dituning.
- ROC-AUC 0.8296 tetap **di bawah syarat awal 0,85**; CI95 [0.6767, 0.9549] memuat 0,85.
- Evaluasi hanya 7 NIM / 40 file; kesalahan menumpuk pada NIM 24-168.
- Fungsi mengasumsikan audio berdurasi mendekati 30 detik; durasi ekstrem dikenali lewat crop/pad tetapi kualitas prediksi menurun bila bahan utama terbuang.
- Rekaman berisi suara orang: gunakan hanya untuk proyek pembelajaran; tanpa pengenalan pembicara.

## Kesimpulan dan arah lanjut

1. Deployment selesai: `predict_audio(path)` terbukti identik prapemrosesannya untuk 176 file dan mereproduksi evaluasi Tahap 05 untuk 40 file uji.
2. `artifacts/LAPORAN.md` berisi seluruh angka aktual, lokasi artefak, dan limitasi untuk konsumen laporan.
3. Bila kelayakan 0,85 ingin dipenuhi, langkah berikutnya adalah siklus data/model baru (data lebih banyak dan lebih beragam), bukan penilaian ulang pada hold-out yang sama.

**Deployment selesai dan tervalidasi. Tanpa training ulang; metrik dilaporkan apa adanya.**


## Kesimpulan dan status akhir proyek

1. **Deployment selesai tanpa training ulang.** `predict_audio(path)` dibangun dari `artifacts/best_model.joblib` (sidik jari file `964a1550c9c06123...`); probabilitas langsung dari `features.csv` identik dengan `test_predictions.csv` Tahap 05 (selisih **1,1e-16**), sehingga model yang dipakai terbukti model yang sama.
2. **Prapemrosesan terbukti identik.** Seluruh **176 file** diperiksa ulang lewat `predict_audio`: selisih fitur relatif maksimum **6,1e-05** (hanya kuantisasi PCM16 ulang, toleransi 1e-03), label cocok **176/176** terhadap pipeline langsung, dan 40 file uji **40/40** cocok dengan Tahap 05 (selisih probabilitas maksimum **2,3e-05**); accuracy 40 file uji **0,7750** sama persis dengan Tahap 05.
3. **Demo nyata berjalan:** `24-068_berisik_1.mp3` → **berisik (p = 0,9114)** benar; `24-068_non-berisik_1.mp3` → **berisik (p = 0,7403)** keliru — konsisten dengan kesalahan FP yang sama pada Tahap 05, dilaporkan apa adanya; file 10 detik teruji pada cabang **pad** dan file 60 detik pada cabang **crop**.
4. **Laporan akhir `artifacts/LAPORAN.md`** memuat data aktual (176 file, 34 NIM, 88:88), 12 fitur + 8 fitur terpilih, model dan konfigurasinya, metrik hold-out nyata **ROC-AUC 0,8296 — DI BAWAH SYARAT AWAL 0,85** dengan CI95 [0,6767; 0,9549], cara pemakaian `predict_audio`, lokasi seluruh artefak, serta limitasi dan etika.
5. **Validasi akhir Tahap 06: 31/31 pemeriksaan PASS** (`artifacts/06_deployment/validation_report.md`); `raw_audio/` tetap 200 file dan `data_final/` tetap 176 WAV.
6. **Status CRISP-DM:** Tahap 01-06 seluruhnya selesai dan tervalidasi. Peningkatan menuju syarat ROC-AUC 0,85 hanya lewat siklus data/model baru — bukan penalaan pada hold-out yang sudah dipakai.

**Deployment selesai dan tervalidasi. Seluruh tahap 01-06 selesai; angka dilaporkan apa adanya.**
